# TP7 : ajouter un broker, réassigner, redémarrer sans interruption, décommissionner

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 14:00-15:10, après la démo du TP (13:55-14:00) |
| Compétences | C14 Réassigner des partitions, ajouter et retirer un broker, ajouter un log dir ; C15 Rolling restart sans interruption de service |
| Slides de rappel | S61 (log dirs), S64 (configuration statique et dynamique), S65 (réassignation), S66 (ajout), S67 (décommissionnement), S68 (rolling restart) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une **zone `# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- Le client Python n'a pas d'API pour la réassignation, les répertoires de données (log dirs), le quorum ni le désenregistrement d'un broker. Dans ce TP, votre code Python **pilote les outils Java** par des cellules `!docker exec kafka-tools ...` et **exploite leurs sorties** : les plans de réassignation et l'état des log dirs sont en JSON.
- Les plans sont écrits dans `/tmp` du conteneur `tools`, comme dans la version Markdown.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, ecrites ici : elles ne font pas partie de confluent-kafka
def attendre(condition, timeout=60, pas=2, message="condition"):
    """Rappelle condition() jusqu'a obtenir une valeur vraie, et la renvoie."""
    debut = time.time()
    while True:
        try:
            valeur = condition()
        except Exception:
            valeur = None
        if valeur:
            return valeur
        if time.time() - debut >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(pas)

def tableau(lignes, entetes):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in ligne) + " |" for ligne in lignes]
    display(Markdown("\n".join(md)))

In [ ]:
import json
import os
import shutil
import threading
import time
from collections import Counter

from confluent_kafka import ElectionType, KafkaException, Producer
from confluent_kafka.admin import AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource, NewTopic


def placement(topics=None):
    """{(topic, partition): (leader, [repliques])}, pour les topics donnes ou pour tous les topics."""
    admin = AdminClient(CONF)
    meta = admin.list_topics(timeout=10).topics
    return {(t, p.id): (p.leader, p.replicas) for t in (topics or meta) for p in meta[t].partitions.values()}


def repliques_par_broker(topics=None):
    """Nombre de repliques hebergees par chaque broker."""
    return dict(sorted(Counter(b for _, repliques in placement(topics).values() for b in repliques).items()))


def urp():
    """Nombre de partitions sous-repliquees."""
    admin = AdminClient(CONF)
    return sum(1 for t in admin.list_topics(timeout=10).topics.values()
               for p in t.partitions.values() if len(p.isrs) < len(p.replicas))


def log_dirs(broker, topics=None):
    """{repertoire: {partition: taille}} d'un broker, lu dans la sortie JSON de kafka-log-dirs.sh."""
    options = " --topic-list " + ",".join(topics) if topics else ""
    sortie = !docker exec kafka-tools bash -c "kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list {broker}{options} | tail -1" 2>&1
    sortie = sortie.n
    return {d["logDir"]: {p["partition"]: p["size"] for p in d["partitions"]}
            for b in json.loads(sortie)["brokers"] for d in b["logDirs"]}


def sain(noeud):
    sortie = !docker compose ps --format '{{{{.Health}}}}' kafka-{noeud} 2>&1
    return sortie.n == "healthy"


admin = AdminClient(CONF)
print("Brokers :", sorted(n.id for n in admin.describe_cluster().result().nodes), "- URP :", urp())

## 1. Objectifs

À la fin du TP, vous savez :

- ajouter un broker à un cluster KRaft et vérifier son enregistrement ;
- produire, exécuter et suivre un plan de réassignation, avec une limite de débit ;
- rééquilibrer les leaders après une réassignation ;
- ajouter un répertoire de données à un nœud et y déplacer une réplique ;
- réaliser un rolling restart sous charge, sans perte ni échec côté producteur ;
- retirer un broker proprement : cordon, réassignation, arrêt, désenregistrement.

## 2. Durée

70 minutes, après 5 minutes de démonstration.

| Étape | Durée |
|---|---|
| 7.1 Démarrer kafka-4 | 8 min |
| 7.2 Constater la répartition | 5 min |
| 7.3 Premier cycle de réassignation | 10 min |
| 7.4 Étendre `orders` et `payments` à kafka-4 | 8 min |
| 7.5 Ajouter un log dir (défi) | 12 min |
| 7.6 Rolling restart sous charge | 14 min |
| 7.7 Décommissionner kafka-4 | 13 min |

## 3. Prérequis

- Lab en mode `extended`, sain : `scripts/validate.sh`. Lab démarré par `scripts/start.sh` (sinon `docker compose` ne peut pas créer `kafka-4`).
- Topics `orders` (6 partitions) et `payments` (3 partitions) présents, issus des TP2 et TP4.
- Aucune partition sous-répliquée.
- Notions : slides S61 à S68.

## 4. Architecture utilisée

```
  Avant                                  Pendant le TP                         Après 7.7
  kafka-1  broker+contrôleur  az-a       kafka-1  data + data2 (7.5)            kafka-1  data + data2
  kafka-2  broker+contrôleur  az-b       kafka-2                                kafka-2
  kafka-3  broker+contrôleur  az-c       kafka-3                                kafka-3
                                         kafka-4  broker seul, az-a             (retiré, désenregistré)
                                                  observateur du quorum
```

| Nœud | Rôles | Rack | Remarque |
|---|---|---|---|
| kafka-1 à kafka-3 | broker, controller (votants) | az-a, az-b, az-c | Quorum de 3 votants |
| kafka-4 | broker seul | az-a | Profil Compose `scale` ; formaté avec `--no-initial-controllers` |

| Lab | Production |
|---|---|
| Nœuds combinés broker et contrôleur | Contrôleurs dédiés (3 ou 5), sur leurs propres machines |
| Le nouveau répertoire est formaté par le script de démarrage (`format --ignore-formatted`) | Formatage du disque ajouté : étape manuelle de la procédure, avant le redémarrage |
| Plans générés par `--generate`, qui ne tient compte que du nombre de répliques | Outil de rééquilibrage qui tient compte de la charge (Cruise Control, par exemple) |

## 5. Contexte métier

Le Black Friday approche. NovaShop ajoute un quatrième broker et doit y répartir les topics critiques sans interrompre les ventes. Le disque de `kafka-1` est presque plein : un second disque est installé. Un correctif de sécurité impose ensuite un redémarrage de tous les nœuds en pleine journée. Enfin, après l'événement, le broker temporaire est rendu.

## 6. Énoncé

1. Démarrez `kafka-4` et prouvez qu'il fait partie du cluster.
2. Montrez qu'il ne reçoit aucune partition existante.
3. Répartissez `orders-history`, puis `orders` et `payments`, sur les 4 brokers, avec une limite de débit.
4. Ajoutez le répertoire `/var/lib/kafka/data2` à `kafka-1` et déplacez-y une réplique de `payments`.
5. Redémarrez les 3 nœuds d'origine un par un, sous un producteur `acks=all`, sans aucun envoi en échec.
6. Retirez `kafka-4` du cluster sans perdre de réplique.

## 7. Étapes

### 7.1 Démarrer kafka-4

**Exemple en ligne de commande** : les paramètres clés du nouveau nœud.

In [ ]:
!grep -E '^(process.roles|node.id|controller.quorum.bootstrap.servers|broker.rack|log.dirs)=' config/kafka/kafka-4/server.properties

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile scale up -d kafka-4
```

In [ ]:
# TODO 7.1 : demarrer kafka-4 et prouver qu'il est enregistre
# 1. Demarrez le noeud : `!docker compose --profile scale up -d kafka-4`.
# 2. Attendez (attendre, timeout=90) que le broker 4 figure dans admin.describe_cluster().result().nodes.
# 3. brokers : {identifiant: rack} des noeuds de describe_cluster() (attributs id et rack).
# 4. traces_demarrage : lignes de `!docker compose logs kafka-4` qui contiennent start-node, Formatting
#    ou Kafka Server started (grep -E).
brokers = {}
traces_demarrage = ""

print(brokers)

In [ ]:
# Verification 7.1
assert brokers == {1: "az-a", 2: "az-b", 3: "az-c", 4: "az-a"}, "A revoir : 4 brokers, kafka-4 dans le rack az-a"
print("OK  4 brokers, kafka-4 dans le rack az-a")
assert "Formatting metadata directory" in traces_demarrage and "Kafka Server started" in traces_demarrage, "A revoir : kafka-4 formate puis demarre"
print("OK  kafka-4 formate puis demarre")

Statut dans le quorum et contenu du répertoire de données de `kafka-4` (lecture seule) :

In [ ]:
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication
!docker exec kafka-tools kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list 4 | tail -1

**Résultat attendu** :

- `process.roles=broker`, `node.id=4`, `controller.quorum.bootstrap.servers` identique aux autres nœuds, `broker.rack=az-a` ;
- traces : `[start-node] Formatage du stockage (cluster W5FmqobtRp-BMcXlZhVKBw)`, `Formatting metadata directory /var/lib/kafka/data with metadata.version 4.3-IV0.`, puis `Kafka Server started` ;
- 4 brokers ; dans le quorum, 3 lignes `Leader` ou `Follower`, et une ligne `4 ... Observer` avec un lag de 0 ;
- répertoire de `kafka-4` vide : `"logDir":"/var/lib/kafka/data"` avec `"partitions":[]`.

### 7.2 Constater la répartition

Les fonctions `placement()` et `repliques_par_broker()` de la cellule d'outils lisent le placement des répliques.

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic orders-history --partitions 4 --replication-factor 3
kafka-producer-perf-test.sh --bootstrap-server $BS --topic orders-history \
  --num-records 20000 --record-size 500 --throughput -1
```

In [ ]:
# TODO 7.2 : repartition avant et apres un nouveau topic
# 1. avant : repliques_par_broker(["orders", "payments"])
# 2. Creez orders-history (4 partitions, RF 3) et ecrivez-y 20000 messages de 500 octets avec un Producer
#    (b"x" * 500 ; poll(0) apres chaque produce, et en cas de BufferError, poll(0.1) puis nouvel essai ; flush() a la fin).
# 3. historique : repliques_par_broker(["orders-history"])
avant = historique = {}

print("orders et payments :", avant, "- orders-history :", historique)

In [ ]:
# Verification 7.2
assert 4 not in avant, "A revoir : aucune replique existante sur le broker 4"
print("OK  aucune replique existante sur le broker 4")
assert historique.get(4, 0) > 0, "A revoir : le nouveau topic utilise le broker 4 : %s" % historique
print("OK  le nouveau topic utilise le broker 4 : %s" % historique)

**Résultat attendu** : aucune réplique d'`orders` ni de `payments` sur le broker 4 ; `orders-history` en a plusieurs. Un nouveau topic utilise tous les brokers disponibles, les anciens ne bougent pas.

### 7.3 Premier cycle de réassignation

`kafka-reassign-partitions.sh --generate` propose un plan à partir d'une liste de topics et de brokers. Sa sortie contient deux lignes JSON, chacune après un titre : `Current partition replica assignment` (la répartition actuelle, votre retour arrière) et `Proposed partition reassignment configuration` (le plan proposé).

Les deux fonctions ci-dessous sont fournies : `executer(plan, debit)` écrit le plan dans `/tmp/plan.json` du conteneur `tools` et lance `--execute` (avec `--throttle` si un débit est donné) ; `verifier_plan()` lance `--verify` sur ce fichier.

**Exemple en ligne de commande** : un plan généré pour `orders-history` (lecture seule : `--generate` ne change rien).

In [ ]:
def executer(plan, debit=None):
    !echo '{json.dumps(plan)}' | docker exec -i kafka-tools tee /tmp/plan.json > /dev/null
    option = " --throttle %d" % debit if debit else ""
    sortie = !docker exec kafka-tools bash -c "kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --execute{option}" 2>&1
    return sortie.n


def verifier_plan():
    sortie = !docker exec kafka-tools kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --verify 2>&1
    return sortie.n


!echo '{{"version":1,"topics":[{{"topic":"orders-history"}}]}}' | docker exec -i kafka-tools tee /tmp/topics.json
!docker exec kafka-tools bash -c "kafka-reassign-partitions.sh --bootstrap-server $BS --topics-to-move-json-file /tmp/topics.json --broker-list 1,2,3,4 --generate | cut -c1-200"

In [ ]:
# TODO 7.3 a : les fonctions generer() et limitations_broker(), puis un premier plan limite
# 1. generer(topics, brokers) : ecrit {"version": 1, "topics": [{"topic": t}, ...]} dans /tmp/topics.json du
#    conteneur tools (`!echo '...' | docker exec -i kafka-tools tee /tmp/topics.json`, avec json.dumps), lance --generate avec
#    --broker-list construit a partir de brokers ("1,2,3,4"), puis renvoie (actuel, propose) : les deux lignes JSON
#    qui suivent les titres, lues avec json.loads (lignes = sortie.splitlines() ; lignes.index(titre) + 1).
# 2. limitations_broker(broker) : {nom: valeur} des entrees de describe_configs sur ConfigResource("broker", str(broker))
#    dont le nom contient "throttled" et dont la source est ConfigSource.DYNAMIC_BROKER_CONFIG.value.
# 3. retour_arriere, plan = generer(["orders-history"], [1, 2, 3, 4]) ; executer(plan, 50000000) ;
#    aussitot apres, limitations_4 = limitations_broker(4).
def generer(topics, brokers):
    return {}, {}


def limitations_broker(broker):
    return {}


retour_arriere = plan = {}
limitations_4 = {}

print("partitions du plan :", len(plan.get("partitions", [])), "- limitations sur le broker 4 :", limitations_4)

In [ ]:
# Verification 7.3 a
assert len(plan.get("partitions", [])) == 4 and len(retour_arriere.get("partitions", [])) == 4, "A revoir : plan et retour arriere pour les 4 partitions"
print("OK  plan et retour arriere pour les 4 partitions")
assert limitations_4.get("leader.replication.throttled.rate") == "50000000", "A revoir : limitation posee sur le broker 4"
print("OK  limitation posee sur le broker 4")

Équivalent en ligne de commande du `TODO` b (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --verify
kafka-leader-election.sh --bootstrap-server $BS --election-type preferred --all-topic-partitions
```

In [ ]:
# TODO 7.3 b : suivre jusqu'a la fin, puis reelire les leaders
# 1. attendre_fin(timeout) : relance verifier_plan() toutes les 5 s (attendre, pas=5) jusqu'a ce que sa sortie
#    contienne "Clearing topic-level throttles", et renvoie cette derniere sortie (attendre() renvoie la valeur
#    vraie obtenue : faites renvoyer None a la condition tant que ce n'est pas fini).
# 2. fin_73 = attendre_fin(120) ; limitations_apres = limitations_broker(4).
# 3. non_preferes : nombre de partitions d'orders-history dont le leader n'est pas le premier de la liste des repliques.
# 4. Lancez l'election des leaders preferes (elect_leaders, comme au TP4), puis recalculez non_preferes_apres.
def attendre_fin(timeout):
    return ""


fin_73 = ""
limitations_apres = None
non_preferes = non_preferes_apres = None

print("leaders non preferes avant l'election :", non_preferes, "- apres :", non_preferes_apres)

In [ ]:
# Verification 7.3 b
assert "Clearing broker-level throttles" in fin_73, "A revoir : --verify a retire les limitations"
print("OK  --verify a retire les limitations")
assert limitations_apres == {}, "A revoir : plus aucune limitation sur le broker 4"
print("OK  plus aucune limitation sur le broker 4")
assert non_preferes_apres == 0, "A revoir : chaque partition d'orders-history a son leader prefere"
print("OK  chaque partition d'orders-history a son leader prefere")

**Résultat attendu** :

- le plan proposé utilise `"log_dirs":["any","any","any"]` ; `--execute` répond `The inter-broker throttle limit was set to 50000000 B/s` ;
- juste après : `leader.replication.throttled.rate` et `follower.replication.throttled.rate` à 50000000 sur le broker 4 ;
- `--verify` : `Reassignment of partition orders-history-N is completed.` pour les 4 partitions, puis `Clearing broker-level throttles on brokers 1,2,3,4` et `Clearing topic-level throttles on topic orders-history` ;
- après réassignation, certains leaders ne sont plus le premier de leur liste ; l'élection les remet en tête.

### 7.4 Étendre `orders` et `payments` à kafka-4

Même cycle, avec vos fonctions, et une limite de 1 Mo/s. La fonction `log_dirs(broker, topics)` de la cellule d'outils lit la taille des répliques d'un broker.

In [ ]:
# TODO 7.4 : orders et payments sur les 4 brokers
# 1. retour_74, plan_74 = generer(["orders", "payments"], [1, 2, 3, 4]) ; executer(plan_74, 1000000).
# 2. premier_verify : la sortie de verifier_plan() aussitot apres.
# 3. Attendez la fin : attendre_fin(180).
# 4. sur_4 : log_dirs(4, ["orders", "payments"]) ; repartition_74 : repliques_par_broker(["orders", "payments"]).
premier_verify = ""
sur_4 = repartition_74 = {}

print("sur le broker 4 :", sur_4)
print("repartition :", repartition_74)

In [ ]:
# Verification 7.4
assert "Status of partition reassignment" in premier_verify, "A revoir : suivi lance aussitot apres --execute"
print("OK  suivi lance aussitot apres --execute")
assert repartition_74.get(4, 0) > 0 and limitations_broker(4) == {}, "A revoir : repliques sur le broker 4, limitations retirees"
print("OK  repliques sur le broker 4, limitations retirees")

**Résultat attendu** : `The inter-broker throttle limit was set to 1000000 B/s`. Selon le volume des partitions, le premier `--verify` montre des partitions `still in progress` ; avec des partitions très petites, tout peut être `completed` dès le premier `--verify`. Le broker 4 héberge ensuite des répliques des deux topics, avec leur taille en octets.

### 7.5 Défi : ajouter un log dir

Le paramètre `log.dirs` de `config/kafka/kafka-1/server.properties` liste les répertoires de données du nœud, séparés par des virgules. Le fichier est lisible et modifiable depuis le notebook (`FICHIER_KAFKA_1`). C'est une configuration **statique** : le nœud doit redémarrer pour la prendre en compte.

Un plan de réassignation peut aussi déplacer une réplique **entre les répertoires d'un même broker** : `log_dirs` contient un élément par réplique, dans le même ordre que `replicas`, et la valeur `"any"` laisse le broker choisir.

```json
{"version":1,"partitions":[{"topic":"payments","partition":<n>,"replicas":[<répliques actuelles, dans l'ordre>],"log_dirs":[<un élément par réplique>]}]}
```

In [ ]:
FICHIER_KAFKA_1 = os.path.join(LAB_DIR, "config/kafka/kafka-1/server.properties")
!grep '^log.dirs' config/kafka/kafka-1/server.properties
print(log_dirs(1, ["payments"]))

Équivalent en ligne de commande du `TODO` a (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
sed -i.bak 's|^log.dirs=/var/lib/kafka/data$|log.dirs=/var/lib/kafka/data,/var/lib/kafka/data2|' \
  config/kafka/kafka-1/server.properties
docker compose restart kafka-1
```

In [ ]:
# TODO 7.5 a : ajouter /var/lib/kafka/data2 a kafka-1
# 1. Copiez FICHIER_KAFKA_1 en FICHIER_KAFKA_1 + ".bak" (shutil.copyfile), puis remplacez dans le fichier la ligne
#    log.dirs=/var/lib/kafka/data par log.dirs=/var/lib/kafka/data,/var/lib/kafka/data2.
# 2. Redemarrez kafka-1 (`!docker compose restart kafka-1`), puis attendez qu'il soit sain (sain(1)) et que urp() vaille 0.
# 3. traces_formatage : lignes des traces recentes de kafka-1 qui contiennent start-node ou Formatting
#    (docker compose logs kafka-1 --since 2m).
# 4. repertoires_1 : liste triee des repertoires de log_dirs(1).
traces_formatage = ""
repertoires_1 = []

print(repertoires_1)

In [ ]:
# Verification 7.5 a
assert "Formatting data directory /var/lib/kafka/data2" in traces_formatage, "A revoir : data2 formate au demarrage"
print("OK  data2 formate au demarrage")
assert repertoires_1 == ["/var/lib/kafka/data", "/var/lib/kafka/data2"], "A revoir : kafka-1 a deux repertoires"
print("OK  kafka-1 a deux repertoires")

In [ ]:
# TODO 7.5 b (defi) : deplacer une seule replique de payments vers data2
# 1. Choisissez une partition de payments dont kafka-1 heberge une replique : placement(["payments"]).
# 2. plan_logdir : le plan d'une seule partition, avec les repliques actuelles dans l'ordre et "log_dirs" qui
#    place la replique du broker 1 dans /var/lib/kafka/data2 ("any" pour les autres).
# 3. executer(plan_logdir), puis attendez que verifier_plan() contienne "completed successfully".
# 4. apres_logdir : log_dirs(1, ["payments"]).
plan_logdir = {}
apres_logdir = {}

print(json.dumps(plan_logdir))
print(apres_logdir)

In [ ]:
# Verification 7.5 b
deplacees = list(apres_logdir.get("/var/lib/kafka/data2", {}))
assert len(deplacees) == 1 and deplacees[0].startswith("payments-"), "A revoir : une replique de payments dans data2 : %s" % deplacees
print("OK  une replique de payments dans data2 : %s" % deplacees)
assert not set(deplacees) & set(apres_logdir.get("/var/lib/kafka/data", {})), "A revoir : elle n'est plus dans data"
print("OK  elle n'est plus dans data")

In [ ]:
!docker compose exec -T kafka-1 ls /var/lib/kafka/data2 </dev/null

**Résultat attendu** :

- traces : `Formatting data directory /var/lib/kafka/data2 with metadata.version 4.3-IV0.` ; le répertoire d'origine n'est pas reformaté ;
- `--execute` : `Successfully started moving log directory to /var/lib/kafka/data2 for replica payments-N with broker 1` ;
- `--verify` : `Reassignment of replica payments-N-1 completed successfully.` ;
- la partition apparaît sous `/var/lib/kafka/data2`, et plus sous `/var/lib/kafka/data`.

### 7.6 Rolling restart sous charge

La classe `Charge` du TP4 revient : un producteur en arrière-plan, ici `acks=all` et idempotent. Pendant qu'il écrit 90 000 messages à 300 messages/s (5 minutes), redémarrez les nœuds 1 à 3 **un par un**, en attendant entre deux nœuds qu'il soit sain et que l'URP revienne à 0.

In [ ]:
class Charge:
    """Producteur de charge en arriere-plan : nombre messages, a debit messages par seconde."""

    def __init__(self, topic, nombre, debit, config):
        self.confirmes = 0
        self.erreurs = []
        self.producteur = Producer({"bootstrap.servers": BS, "log_level": 4, **config})
        self._fil = threading.Thread(target=self._envoyer, args=(topic, nombre, debit), daemon=True)
        self._fil.start()

    def _rapport(self, err, msg):
        if err is None:
            self.confirmes += 1
        else:
            self.erreurs.append(err)

    def _envoyer(self, topic, nombre, debit):
        debut = time.time()
        for i in range(nombre):
            attente = debut + i / debit - time.time()
            if attente > 0:
                time.sleep(attente)
            self.producteur.produce(topic, value=b"x" * 200, on_delivery=self._rapport)
            self.producteur.poll(0)
        self.producteur.flush(120)

    def en_cours(self):
        return self._fil.is_alive()

    def __repr__(self):
        return "Charge(%d confirmes, %d erreurs%s)" % (self.confirmes, len(self.erreurs),
                                                      ", en cours" if self.en_cours() else ", terminee")

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**), pour chaque nœud `N` :

```bash
# [hôte]
docker compose restart kafka-N
docker compose ps kafka-N        # jusqu'à (healthy)
# [tools]
kafka-topics.sh --bootstrap-server $BS --describe --under-replicated-partitions | wc -l   # jusqu'à 0
```

**Exécutez la cellule `TODO` une seule fois** : elle dure environ 5 minutes.

In [ ]:
# TODO 7.6 : rolling restart sous charge
# 1. leader_quorum : le controleur actif, entier lu apres "LeaderId:" dans
#    `sortie = !docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status` (sortie.n joint les lignes).
#    ordre : liste des noeuds 1, 2, 3, avec leader_quorum en dernier.
# 2. charge = Charge("payments", 90000, 300, {"acks": "all", "enable.idempotence": True})
# 3. Pour chaque noeud de ordre : `!docker compose restart kafka-{n}`, puis attendez sain(n) et urp() == 0
#    (timeout=120) ; ajoutez a journal le tuple (noeud, secondes depuis le debut, charge.confirmes).
# 4. Attendez la fin de la charge (attendre(lambda: not charge.en_cours(), timeout=360, pas=5)).
# 5. Relancez l'election des leaders preferes.
leader_quorum = None
ordre = []
journal = []
charge = None

tableau(journal, ["Noeud", "Sain, URP a 0 (s)", "Messages confirmes"])
print(charge)

In [ ]:
# Verification 7.6
assert sorted(ordre) == [1, 2, 3] and ordre[-1] == leader_quorum and len(journal) == 3, "A revoir : 3 noeuds redemarres, controleur actif (%s) en dernier : %s" % (leader_quorum, ordre)
print("OK  3 noeuds redemarres, controleur actif (%s) en dernier : %s" % (leader_quorum, ordre))
assert charge is not None and charge.confirmes == 90000 and not charge.erreurs, "A revoir : 90000 messages confirmes, aucune erreur : %s" % charge
print("OK  90000 messages confirmes, aucune erreur : %s" % charge)
assert urp() == 0, "A revoir : aucune partition sous-repliquee"
print("OK  aucune partition sous-repliquee")

**Résultat attendu** :

- le contrôleur actif est redémarré en dernier ;
- chaque nœud redevient sain en 10 à 20 s, et l'URP revient à 0 en quelques secondes ;
- 90 000 messages confirmés, aucune erreur définitive : le producteur a suivi les changements de leader.

### 7.7 Décommissionner kafka-4

Le **cordon** (`cordoned.log.dirs=*`, configuration dynamique du broker) empêche toute nouvelle réplique d'arriver sur le broker. Ensuite : réassigner ses répliques vers les autres brokers, l'arrêter, puis le désenregistrer.

Équivalent en ligne de commande du `TODO` a (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type brokers --entity-name 4 \
  --add-config 'cordoned.log.dirs=*'
```

In [ ]:
# TODO 7.7 a : cordon et reassignation vers 1, 2, 3
# 1. Cordonnez le broker 4 en Python : incremental_alter_configs sur ConfigResource("broker", "4") avec
#    ConfigEntry("cordoned.log.dirs", "*", incremental_operation=AlterConfigOpType.SET) (comme au TP4).
# 2. topics_sur_4 : liste triee des topics qui ont une replique sur le broker 4 (placement()).
# 3. retour_77, plan_77 = generer(topics_sur_4, [1, 2, 3]) ; brokers_du_plan : ensemble des brokers cites dans plan_77.
# 4. executer(plan_77, 50000000), puis attendre_fin(180). reste_sur_4 : log_dirs(4).
topics_sur_4 = []
brokers_du_plan = set()
reste_sur_4 = None

print(topics_sur_4, brokers_du_plan, reste_sur_4)

In [ ]:
# Verification 7.7 a
assert brokers_du_plan == {1, 2, 3}, "A revoir : le plan n'utilise que les brokers 1, 2 et 3"
print("OK  le plan n'utilise que les brokers 1, 2 et 3")
assert reste_sur_4 == {"/var/lib/kafka/data": {}}, "A revoir : le broker 4 n'heberge plus aucune replique"
print("OK  le broker 4 n'heberge plus aucune replique")

Équivalent en ligne de commande du `TODO` b (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile scale stop kafka-4
# [tools]
kafka-cluster.sh unregister --bootstrap-server $BS --id 4
```

In [ ]:
# TODO 7.7 b : arreter puis desenregistrer
# 1. Arretez kafka-4 (`!docker compose --profile scale stop kafka-4`).
# 2. Affichez le quorum (kafka-metadata-quorum.sh ... describe --replication) : kafka-4 y figure encore.
# 3. desenregistrement : sortie de `!docker exec kafka-tools kafka-cluster.sh unregister --bootstrap-server $BS --id 4`.
# 4. brokers_finaux : liste triee des identifiants de describe_cluster().
desenregistrement = ""
brokers_finaux = []

print(desenregistrement, brokers_finaux)

In [ ]:
# Verification 7.7 b
assert "no longer registered" in desenregistrement, "A revoir : broker 4 desenregistre"
print("OK  broker 4 desenregistre")
assert brokers_finaux == [1, 2, 3], "A revoir : brokers 1, 2, 3"
print("OK  brokers 1, 2, 3")
assert not [k for k, (_, r) in placement().items() if 4 in r] and urp() == 0, "A revoir : aucune replique orpheline, URP a 0"
print("OK  aucune replique orpheline, URP a 0")

**Résultat attendu** :

- topics sur le broker 4 : `orders`, `orders-history`, `payments` ; le plan ne contient que les brokers 1, 2 et 3 ;
- après la réassignation, le répertoire de `kafka-4` est vide ;
- après l'arrêt, le quorum montre encore le nœud 4 en `Observer`, avec un lag qui ne bouge plus ;
- `Broker 4 is no longer registered.` ; restent les brokers 1, 2 et 3, sans partition sous-répliquée.

## Bilan de l'exercice

In [ ]:
tableau([
    ("kafka-4 enregistre puis retire", "%s puis %s" % (sorted(brokers), brokers_finaux), "7.1, 7.7"),
    ("Repliques sur le broker 4 apres 7.4", repartition_74.get(4, "a completer") if repartition_74 else "a completer", "7.4"),
    ("Repertoires de kafka-1", ", ".join(repertoires_1) or "a completer", "7.5"),
    ("Rolling restart : ordre, messages confirmes", "%s, %s" % (ordre, charge), "7.6"),
    ("Partitions sous-repliquees", urp(), "fin"),
], ["Controle", "Valeur", "Etape"])

## 11. Questions de réflexion

1. Pourquoi un broker ajouté ne reçoit-il aucune partition existante ? Que se passe-t-il si l'on oublie de réassigner ?
2. Pourquoi conserver la répartition actuelle affichée par `--generate` ? Comment l'utiliseriez-vous ?
3. Que se passe-t-il si l'on oublie de lancer `--verify` jusqu'au bout après un `--throttle` ?
4. Pourquoi kafka-4 est-il `Observer` et non `Follower` dans le quorum ?
5. Une partition vit dans un seul log dir. Quel impact sur l'équilibre entre les disques ? Que se passe-t-il si le disque `data2` tombe en panne ?
6. Pourquoi redémarrer le contrôleur actif en dernier ?
7. Le producteur n'a reçu aucune erreur. Pourquoi ? Qu'est-ce qui aurait changé avec un topic en RF 2 et `min.insync.replicas=2` ?
8. Pourquoi cordonner le broker **avant** de réassigner ? Pourquoi l'arrêter **avant** de le désenregistrer ?

## 12. Erreurs fréquentes

| Symptôme | Cause | Correction |
|---|---|---|
| `ValueError: ... is not in list` dans `generer()` | `--generate` en erreur : titre absent de la sortie | Afficher la sortie complète (`print(sortie.n)`) et lire l'erreur |
| `--verify` reste longtemps `still in progress` | Limite de débit trop basse pour le volume | `--execute --additional --throttle <plus haut>` avec le même plan, puis `--verify` |
| Le broker garde `leader.replication.throttled.rate` | `--verify` jamais relancé après la fin | Relancer `verifier_plan()` |
| Leaders déséquilibrés après réassignation | Le leader n'est pas le premier de la liste `Replicas` | `elect_leaders(ElectionType.PREFERRED, None)` |
| La réplique ne va pas dans `data2` | Ordre de `log_dirs` différent de celui de `replicas` | Un élément par réplique, à la position du broker 1 |
| Erreurs `NOT_ENOUGH_REPLICAS` pendant le rolling restart | Nœud suivant redémarré avant que l'URP soit revenue à 0 : deux répliques absentes | Toujours attendre `urp() == 0` entre deux nœuds |
| `The manual partition assignment includes broker 4, but all its log directories are cordoned.` | Plan ou création manuelle qui vise un broker cordonné | Normal pendant le décommissionnement ; retirer le broker 4 du plan |
| Le broker 4 réapparaît après `unregister` | Conteneur redémarré : il se réenregistre | Le laisser arrêté |

## 13. Solution formateur

Voir `corrige.ipynb` (et `corrige.md` pour la version en ligne de commande).

## 14. Variante avancée

1. **Augmenter le facteur de réplication.** Créez `audit` avec 3 partitions d'une réplique chacune, sur les brokers 1, 2 et 3 (`NewTopic(..., num_partitions=3, replica_assignment=[[1], [2], [3]])`) et écrivez-y 60 000 messages de 1000 octets. Écrivez à la main le plan qui le passe en RF 3, exécutez-le avec une limite de 100 Ko/s, et observez `Adding Replicas` dans `--describe` et la sortie de `--list`. Annulez avec `--cancel`, vérifiez l'état, puis relancez sans limite.
2. **Effet du cordon.** Pendant l'étape 7.7, avant d'arrêter kafka-4, créez un topic de 12 partitions en RF 3 : combien de répliques arrivent sur le broker 4 ?
3. **Plan minimal.** Le plan de décommissionnement produit par `--generate` déplace-t-il seulement les répliques du broker 4 ? Comptez les partitions modifiées sans nécessité.